In [7]:
import pandas as pd
import os
from pathlib import Path

experiment_logs_path = Path(r"logs\Experiment2and3")

main_results_df = pd.read_csv(experiment_logs_path / "main_results.csv")

main_results_df

,Architecture,Encoder,Pretrained,Parameter Count,Inference Time,Last Validation Dice,Last Validation IoU
0,unet,efficientnet-b0,scratch,6250893.0,8.188087,0.821108,0.727987
1,unet,efficientnet-b0,other-dataset,6250893.0,8.331331,0.944926,0.901546
2,unet,efficientnet-b0,same-dataset,6250893.0,8.491862,0.960552,0.927702
3,unet,efficientnet-b1,scratch,8756529.0,10.957552,0.859526,0.779679
4,unet,efficientnet-b1,other-dataset,8756529.0,11.240377,0.959726,0.925607
...,...,...,...,...,...,...,...
304,unet++,resnet18,other-dataset,15964177.0,7.101258,0.962518,0.932226
305,unet++,resnet18,same-dataset,15964177.0,6.847830,0.959980,0.926910
306,unet++,resnet34,scratch,26072337.0,8.621709,0.944795,0.906387
307,unet++,resnet34,other-dataset,26072337.0,8.477798,0.956499,0.922683


In [8]:
trained_models_path = Path(r"\\zvsl.mvl6.uni-tuebingen.de\Employees\06_Datasets\Bjoern\Segmentation\trained_models\Experiment2and3\architecture_models")

for _, row in main_results_df.iterrows():
    model_name = f"{row['Architecture']}_{row['Encoder']}_zscore_{row['Pretrained']}_final_model.pth"

    model_path = trained_models_path / model_name

    if not model_path.exists():
        print(f"Model {model_name} does not exist at {model_path}")

Model deeplabv3+_efficientnet-b4_zscore_other-dataset_final_model.pth does not exist at \\zvsl.mvl6.uni-tuebingen.de\Employees\06_Datasets\Bjoern\Segmentation\trained_models\Experiment2and3\architecture_models\deeplabv3+_efficientnet-b4_zscore_other-dataset_final_model.pth
Model deeplabv3+_vgg11_bn_zscore_scratch_final_model.pth does not exist at \\zvsl.mvl6.uni-tuebingen.de\Employees\06_Datasets\Bjoern\Segmentation\trained_models\Experiment2and3\architecture_models\deeplabv3+_vgg11_bn_zscore_scratch_final_model.pth
Model deeplabv3+_vgg11_bn_zscore_other-dataset_final_model.pth does not exist at \\zvsl.mvl6.uni-tuebingen.de\Employees\06_Datasets\Bjoern\Segmentation\trained_models\Experiment2and3\architecture_models\deeplabv3+_vgg11_bn_zscore_other-dataset_final_model.pth
Model deeplabv3+_vgg11_bn_zscore_same-dataset_final_model.pth does not exist at \\zvsl.mvl6.uni-tuebingen.de\Employees\06_Datasets\Bjoern\Segmentation\trained_models\Experiment2and3\architecture_models\deeplabv3+_vgg11

In [9]:
import json
import dataset

with open(experiment_logs_path / "participant_split.json", "r") as f:
    participant_split = json.load(f)


path_to_dataset = Path(r"F:\Python\SAM2\OCTDatasetOIMHS")

test_dataset = dataset.OIMHSDataset(
    participants = participant_split["test"],
    path = path_to_dataset,
    augment = False,
    max_rotate_deg = 0,
    return_numpy = False,
    normalize = "zscore"
)

print(f"Number of test samples: {len(test_dataset)}")

Number of test samples: 1243


In [10]:
import torch

def calc_metrics(logits, mask):
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()

    preds_f = preds.view(1, -1)
    masks_f = mask.view(1, -1)

    intersection = (preds_f * masks_f).sum()
    pred_sum = preds_f.sum()
    mask_sum = masks_f.sum()

    union = pred_sum + mask_sum - intersection

    dice_score = (2 * intersection) / (pred_sum + mask_sum)
    iou_score = intersection / union

    return dice_score.item(), iou_score.item()

In [11]:
row

Architecture                  unet++
Encoder                     resnet34
Pretrained              same-dataset
Parameter Count           26072337.0
Inference Time              8.742939
Last Validation Dice        0.955183
Last Validation IoU         0.923167
Name: 308, dtype: object

In [12]:
out_file_path = experiment_logs_path / "metrics_per_image.csv"

def read_combinations(file_path):
    combinations = set()
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                parts = line.split(',')
                # print(parts)
                if len(parts) >= 3:
                    architecture, encoder, pretrained = parts[:3]
                    combinations.add((architecture.strip(), encoder.strip(), pretrained.strip()))
    return combinations

if out_file_path.exists():
    existing_combinations = read_combinations(out_file_path)
print(existing_combinations)

{('unet', 'mobileone_s1', 'same-dataset'), ('unet', 'mit_b1', 'other-dataset'), ('unet', 'mobileone_s0', 'scratch'), ('deeplabv3+', 'efficientnet-b1', 'scratch'), ('unet', 'resnet18', 'scratch'), ('unet', 'efficientnet-b2', 'scratch'), ('Architecture', 'Encoder', 'Pretrained'), ('unet', 'mobileone_s2', 'other-dataset'), ('unet', 'vgg11_bn', 'other-dataset'), ('deeplabv3+', 'efficientnet-b0', 'same-dataset'), ('deeplabv3+', 'efficientnet-b3', 'other-dataset'), ('unet', 'mobileone_s3', 'same-dataset'), ('deeplabv3+', 'efficientnet-b2', 'same-dataset'), ('unet', 'efficientnet-b1', 'same-dataset'), ('unet', 'efficientnet-b4', 'scratch'), ('unet', 'mit_b1', 'same-dataset'), ('unet', 'vgg19_bn', 'other-dataset'), ('unet', 'resnet34', 'scratch'), ('unet', 'vgg13_bn', 'other-dataset'), ('unet', 'mit_b0', 'other-dataset'), ('unet', 'mobilenet_v2', 'other-dataset'), ('unet', 'mobileone_s2', 'same-dataset'), ('unet', 'mobileone_s4', 'other-dataset'), ('unet', 'efficientnet-b0', 'other-dataset'), 

In [14]:
import torch
import segmentation_models_pytorch as smp
from tqdm import tqdm

i = 0
use_gpu = True

device = torch.device("cuda" if torch.cuda.is_available() and use_gpu else "cpu")

out_file_path = experiment_logs_path / "metrics_per_image.csv"

def read_combinations(file_path):
    combinations = set()
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                parts = line.split(',')
                if len(parts) >= 3:
                    architecture, encoder, pretrained = parts[:3]
                    combinations.add((architecture.strip(), encoder.strip(), pretrained.strip()))
    return combinations

if out_file_path.exists():
    combinations = read_combinations(out_file_path)
    writer = open(out_file_path, "a")
else:
    combinations = set()
    writer = open(out_file_path, "w")
    header = "Architecture,Encoder,Pretrained,Participant,File,Dice,IoU\n"
    writer.write(header)

n_models = len(main_results_df)
current_model_n = 0

alias = {
        "unet": "unet",
        "u-net": "unet",
        "unet++": "unetplusplus",
        "unetplusplus": "unetplusplus",
        "deeplabv3+": "deeplabv3plus",
        "deeplabv3plus": "deeplabv3plus",
        "deeplab": "deeplabv3plus",
        "segformer": "segformer",
        "fpn": "fpn",
        "pspnet": "pspnet",
        "linknet": "linknet",
        "pan": "pan",
        "manet": "manet",
        "upernet": "upernet",
        "dpt": "dpt",
    }

for _, row in main_results_df.iterrows():
    current_model_n += 1
    model_name = f"{row['Architecture']}_{row['Encoder']}_zscore_{row['Pretrained']}_final_model.pth"

    if (row['Architecture'], row['Encoder'], row['Pretrained']) in combinations:
        print(f"Skipping model {model_name} as it has already been processed.")
        continue

    model_path = trained_models_path / model_name

    if not model_path.exists():
            print(f"Model {model_name} does not exist at {model_path}")
            continue
    
    model = smp.create_model(
        arch=alias.get(row['Architecture'], row['Architecture']),
        encoder_name=row['Encoder'],
        in_channels=1,
        classes=1,
        activation=None
    )

    
    model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))


    model.to(device)
    model.eval()
    with torch.no_grad():
        for i, (img, mask) in tqdm(enumerate(test_dataset), total=len(test_dataset), desc=f"Model {current_model_n}/{n_models}: Architecture: {row['Architecture']}, Encoder: {row['Encoder']}, Pretrained: {row['Pretrained']}"):
            img_path, _ = test_dataset.samples[i]
            img = img.to(device)
            mask = mask.to(device)
            logits = model(img.unsqueeze(0))
            dice_score, iou_score = calc_metrics(logits, mask)

            folder = img_path.parent.name
            file_name = img_path.name

            # print(f"Sample {i}: Folder: {folder}, File name: {file_name}, Dice score: {dice_score:.4f}, IoU score: {iou_score:.4f}")

            writer.write(f"{row['Architecture']},{row['Encoder']},{row['Pretrained']},{folder},{file_name},{dice_score},{iou_score}\n")
            writer.flush()

writer.close()
    

Skipping model unet_efficientnet-b0_zscore_scratch_final_model.pth as it has already been processed.
Skipping model unet_efficientnet-b0_zscore_other-dataset_final_model.pth as it has already been processed.
Skipping model unet_efficientnet-b0_zscore_same-dataset_final_model.pth as it has already been processed.
Skipping model unet_efficientnet-b1_zscore_scratch_final_model.pth as it has already been processed.
Skipping model unet_efficientnet-b1_zscore_other-dataset_final_model.pth as it has already been processed.
Skipping model unet_efficientnet-b1_zscore_same-dataset_final_model.pth as it has already been processed.
Skipping model unet_efficientnet-b2_zscore_scratch_final_model.pth as it has already been processed.
Skipping model unet_efficientnet-b2_zscore_other-dataset_final_model.pth as it has already been processed.
Skipping model unet_efficientnet-b2_zscore_same-dataset_final_model.pth as it has already been processed.
Skipping model unet_efficientnet-b3_zscore_scratch_final_m

Model 109/309: Architecture: fpn, Encoder: efficientnet-b0, Pretrained: scratch: 100%|██████████| 1243/1243 [00:43<00:00, 28.87it/s]
Model 110/309: Architecture: fpn, Encoder: efficientnet-b0, Pretrained: other-dataset: 100%|██████████| 1243/1243 [00:47<00:00, 26.24it/s]
Model 111/309: Architecture: fpn, Encoder: efficientnet-b0, Pretrained: same-dataset: 100%|██████████| 1243/1243 [00:50<00:00, 24.65it/s]
Model 112/309: Architecture: fpn, Encoder: efficientnet-b1, Pretrained: scratch: 100%|██████████| 1243/1243 [01:00<00:00, 20.58it/s]
Model 113/309: Architecture: fpn, Encoder: efficientnet-b1, Pretrained: other-dataset: 100%|██████████| 1243/1243 [01:03<00:00, 19.48it/s]
Model 114/309: Architecture: fpn, Encoder: efficientnet-b1, Pretrained: same-dataset: 100%|██████████| 1243/1243 [01:05<00:00, 18.87it/s]
Model 115/309: Architecture: fpn, Encoder: efficientnet-b2, Pretrained: scratch: 100%|██████████| 1243/1243 [01:06<00:00, 18.72it/s]
Model 116/309: Architecture: fpn, Encoder: effi

Model unet++_efficientnet-b3_zscore_same-dataset_final_model.pth does not exist at \\zvsl.mvl6.uni-tuebingen.de\Employees\06_Datasets\Bjoern\Segmentation\trained_models\Experiment2and3\architecture_models\unet++_efficientnet-b3_zscore_same-dataset_final_model.pth


Model 283/309: Architecture: unet++, Encoder: efficientnet-b4, Pretrained: scratch: 100%|██████████| 1243/1243 [01:59<00:00, 10.38it/s]
Model 284/309: Architecture: unet++, Encoder: efficientnet-b4, Pretrained: other-dataset: 100%|██████████| 1243/1243 [01:46<00:00, 11.68it/s]
Model 285/309: Architecture: unet++, Encoder: efficientnet-b4, Pretrained: same-dataset: 100%|██████████| 1243/1243 [01:46<00:00, 11.67it/s]


Model unet++_mit_b0_zscore_scratch_final_model.pth does not exist at \\zvsl.mvl6.uni-tuebingen.de\Employees\06_Datasets\Bjoern\Segmentation\trained_models\Experiment2and3\architecture_models\unet++_mit_b0_zscore_scratch_final_model.pth
Model unet++_mit_b0_zscore_other-dataset_final_model.pth does not exist at \\zvsl.mvl6.uni-tuebingen.de\Employees\06_Datasets\Bjoern\Segmentation\trained_models\Experiment2and3\architecture_models\unet++_mit_b0_zscore_other-dataset_final_model.pth
Model unet++_mit_b0_zscore_same-dataset_final_model.pth does not exist at \\zvsl.mvl6.uni-tuebingen.de\Employees\06_Datasets\Bjoern\Segmentation\trained_models\Experiment2and3\architecture_models\unet++_mit_b0_zscore_same-dataset_final_model.pth
Model unet++_mit_b1_zscore_scratch_final_model.pth does not exist at \\zvsl.mvl6.uni-tuebingen.de\Employees\06_Datasets\Bjoern\Segmentation\trained_models\Experiment2and3\architecture_models\unet++_mit_b1_zscore_scratch_final_model.pth
Model unet++_mit_b1_zscore_other-d

Model 292/309: Architecture: unet++, Encoder: mobilenet_v2, Pretrained: scratch: 100%|██████████| 1243/1243 [01:01<00:00, 20.30it/s]
Model 293/309: Architecture: unet++, Encoder: mobilenet_v2, Pretrained: other-dataset: 100%|██████████| 1243/1243 [01:02<00:00, 19.80it/s]
Model 294/309: Architecture: unet++, Encoder: mobilenet_v2, Pretrained: same-dataset: 100%|██████████| 1243/1243 [01:00<00:00, 20.69it/s]
Model 295/309: Architecture: unet++, Encoder: mobileone_s0, Pretrained: scratch: 100%|██████████| 1243/1243 [02:02<00:00, 10.19it/s]
Model 296/309: Architecture: unet++, Encoder: mobileone_s0, Pretrained: other-dataset: 100%|██████████| 1243/1243 [02:06<00:00,  9.81it/s]
Model 297/309: Architecture: unet++, Encoder: mobileone_s0, Pretrained: same-dataset: 100%|██████████| 1243/1243 [02:03<00:00, 10.05it/s]
Model 298/309: Architecture: unet++, Encoder: mobileone_s1, Pretrained: scratch: 100%|██████████| 1243/1243 [01:24<00:00, 14.74it/s]
Model 299/309: Architecture: unet++, Encoder: m